# API reference

Precise information about every public function, type and testing tool. For explanations and workflows, see the
topic guides; this page answers "what exactly does this take and return?".

| Group | Contents |
| --- | --- |
| [Setup](#Setup) | `init`, `shutdown`, `protect`, `protected`, `configure` |
| [Call context](#Call-context) | `site`, `session`, `untrusted` |
| [Verdicts and feedback](#Verdicts-and-feedback) | `verdict`, `feedback`, `on_verdict`, `on_alert` |
| [Introspection](#Introspection) | `status`, `coverage`, `runtime` |
| [Types](#Types) | `Verdict`, `Hit`, `Action`, `Mode`, `Stage`, `Taint`, `SensorInfo`, `SiteStatus` |
| [Exceptions](#Exceptions) | `ImmuneError`, `ConfigError`, `Blocked` |
| [Testing](#Testing:-immune.testing) | `ImmuneHarness`, `FakeReply`, `FakeToolCall`, `FakeProvider`, `MockSensor`, `RecordingSensor`, `ReplaySensor`, `ScenarioLibrary`, `ScenarioRunner`, `LangSmithRecorder` |
| [Command line](#Command-line) | `immune …` |

Everything below is importable from `immune` or `immune.testing`. The signatures on this page are checked against the
code in Immune's test suite; the cell below prints them from the version you have installed.

In [1]:
import inspect

import immune
import immune.testing

for name in [
    "init",
    "shutdown",
    "protect",
    "protected",
    "configure",
    "site",
    "session",
    "untrusted",
    "verdict",
    "feedback",
    "on_verdict",
    "on_alert",
    "status",
    "coverage",
    "runtime",
]:
    print(f"immune.{name}{inspect.signature(getattr(immune, name))}".replace("'", ""))

immune.init(*, mode: str | Mode | None = None, config: str | Path | dict[str, Any] | Settings | None = None, sensor: Sensor | None = None, state_dir: str | Path | None = None, backend: StateBackend | None = None, vaccines: Sequence[str | Path] | None = None) -> Runtime | None
immune.shutdown() -> None
immune.protect(client: Client, **options: Any) -> Client
immune.protected(**options: Any) -> Iterator[Runtime | None]
immune.configure(**changes: Any) -> Settings
immune.site(name: str) -> contextlib.AbstractContextManager[None]
immune.session(session_id: str) -> contextlib.AbstractContextManager[None]
immune.untrusted(text: str, *, source: str = data) -> str
immune.verdict(subject: Any = None) -> Verdict | None
immune.feedback(trace_id: str, label: str, note: str | None = None, threat: str | None = None) -> None
immune.on_verdict(callback: VerdictCallback) -> Callable[[], None]
immune.on_alert(callback: AlertCallback) -> None
immune.status() -> list[SiteStatus]
immune.coverage() -> Cover

## Setup

### `immune.init()`

```python
immune.init(
    *,
    mode: str | Mode | None = None,
    config: str | Path | dict[str, Any] | Settings | None = None,
    sensor: Sensor | None = None,
    state_dir: str | Path | None = None,
    backend: StateBackend | None = None,
    vaccines: Sequence[str | Path] | None = None
) -> Runtime | None
```

Start protecting every LLM call in the process. Call it once at startup, before or after creating clients.

| Parameter | Type | Default | Description |
| --- | --- | --- | --- |
| `mode` | `str \| Mode \| None` | `None` | `"auto"`, `"observe"`, `"strict"` or `"off"`. `None` uses the configuration, then `IMMUNE_MODE`, then `"auto"`. Overrides both when given |
| `config` | `str \| Path \| dict \| Settings \| None` | `None` | A path to an `immune.yaml`, a dict with the same shape, or `Settings`. `None` reads `IMMUNE_CONFIG`, else `./immune.yaml` if it exists, else defaults |
| `sensor` | `Sensor \| None` | `None` | The Jev sensor to use. `None` uses TypeSafe Jev with `TYPESAFE_API_KEY`; tests pass `MockSensor` |
| `state_dir` | `str \| Path \| None` | `None` | Where local state lives. `None` uses `state_dir` from the configuration, else `IMMUNE_HOME`, else `~/.immune` |
| `backend` | `StateBackend \| None` | `None` | A custom state backend; normally configured with `state` instead |
| `vaccines` | `Sequence[str \| Path] \| None` | `None` | Extra vaccine files or folders, added to `vaccines.paths` |

**Returns** `Runtime`: the running instance (also available as `immune.runtime()`), or `None` when `IMMUNE_DISABLED`
is set.

**Raises** `immune.ConfigError` for invalid settings, a missing configuration file, invalid vaccines, or vaccine
switches that would disable a floor rule without `vaccines.allow_floor_changes`.

**Behavior.** Patches the `httpx` and `httpx2` transports used by the OpenAI, Anthropic and Google SDKs, installs
adapters for aiohttp paths, DSPy and boto3 Bedrock, and logs what it covers. Calling it again replaces the running
instance with a new one built from the new arguments.

```python
immune.init()                                   # ./immune.yaml or defaults
immune.init(config="config/immune.yaml")
immune.init(mode="observe", config={"sites": {"support-chat": {"enforce": ["input.off_task"]}}})
```

### `immune.shutdown()`

```python
immune.shutdown() -> None
```

Remove Immune from the process: uninstall the patches, flush callbacks and logs, and close state. Calls made afterwards
are not screened. Safe to call when Immune isn't running.

### `immune.protect()`

```python
immune.protect(client: Client, **options: Any) -> Client
```

Protect one client object instead of the whole process.

| Parameter | Type | Default | Description |
| --- | --- | --- | --- |
| `client` | an OpenAI, Anthropic, Google GenAI or httpx client | required | The client to protect |
| `**options` | | | Passed to `immune.init(...)` (without patching the process) if Immune isn't running yet |

**Returns** the same client, protected. Its own settings (proxies, timeouts, limits, event hooks) are kept.

```python
client = immune.protect(OpenAI(timeout=30))
```

### `immune.protected()`

```python
immune.protected(**options: Any) -> Iterator[Runtime | None]
```

A context manager: `immune.init(**options)` on entry, `immune.shutdown()` on exit. Yields the `Runtime` (or `None` when
`IMMUNE_DISABLED` is set). Useful for scripts and tests.

```python
with immune.protected(mode="strict"):
    run_batch_job()
```

### `immune.configure()`

```python
immune.configure(**changes: Any) -> Settings
```

Change settings while running; applies from the next call. Keyword arguments are merged into the current settings
(nested dicts merge; lists replace).

**Returns** the new `Settings`.

**Raises** `immune.ConfigError` if Immune isn't running, if the result is invalid, or if a setting that decides how
Immune is built changes (`state`, `sensor`, `state_dir`, `canary`, `vaccines.paths`); call `immune.init()` again for
those.

```python
immune.configure(mode="observe")
immune.configure(sites={"support-chat": {"enforce": ["input.off_task"]}})
```

## Call context

### `immune.site()`

```python
immune.site(name: str) -> contextlib.AbstractContextManager[None]
```

A context manager naming the part of your app that makes the calls inside it. Each site gets its own profile and
settings (`sites.<name>` in the configuration).

| Parameter | Type | Description |
| --- | --- | --- |
| `name` | `str` | The site name, such as `"support-chat"` |

**Behavior.** Applies to every LLM call made inside the block, including calls LangChain or LangGraph make in copies
of the context. When the block ends, `immune.verdict()` returns the latest verdict recorded inside it. Nests with
`immune.session()`.

```python
with immune.site("support-chat"):
    response = client.chat.completions.create(...)
```

### `immune.session()`

```python
immune.session(session_id: str) -> contextlib.AbstractContextManager[None]
```

A context manager grouping calls into one conversation. Sessions carry taint, provenance, risk and pending
confirmations across calls.

| Parameter | Type | Description |
| --- | --- | --- |
| `session_id` | `str` | Your conversation, thread or user id |

**Behavior.** Without it, Immune uses the OpenAI Responses chain, the request's `user` field, or a session derived from
the transcript. Like `site`, the latest verdict recorded inside the block is available after it ends.

### `immune.untrusted()`

```python
immune.untrusted(text: str, *, source: str = "data") -> str
```

Mark text you paste into a prompt as untrusted *data*, such as retrieved documents, emails or web pages.

| Parameter | Type | Default | Description |
| --- | --- | --- | --- |
| `text` | `str` | required | The content |
| `source` | `str` | `"data"` | A label for evidence and logs, such as `"kb:returns"` |

**Returns** `str`: the text wrapped in markers. Immune removes the markers before the request is sent, so the model
receives the original text.

```python
prompt = f"Answer from this article:\n{immune.untrusted(article, source='kb:returns')}"
```

## Verdicts and feedback

### `immune.verdict()`

```python
immune.verdict(subject: Any = None) -> Verdict | None
```

Get the verdict for a call.

| `subject` | Looks up |
| --- | --- |
| an SDK response (`ChatCompletion`, `Response`, parsed completion) | by its response id |
| a stream chunk | by its response id, once the stream is finished |
| a LangChain `AIMessage` | by the provider id in `response_metadata` |
| a `str` | by trace id or response id |
| `None` (default) | the latest verdict in the current context, or recorded inside the `site`/`session` block that just ended |

**Returns** `Verdict`, or `None` when the call wasn't screened or is no longer in memory (the most recent 10,000 are
kept).

### `immune.feedback()`

```python
immune.feedback(
    trace_id: str,
    label: str,
    note: str | None = None,
    threat: str | None = None
) -> None
```

Label a verdict, for calibration and reporting.

| Parameter | Type | Default | Description |
| --- | --- | --- | --- |
| `trace_id` | `str` | required | `verdict.trace_id` |
| `label` | `str` | required | `"correct"`, `"false_positive"` or `"missed"` |
| `note` | `str \| None` | `None` | Free text for reviewers |
| `threat` | `str \| None` | `None` | Limit the label to one threat's hit |

**Raises** `immune.ConfigError` for an unknown label, a trace id not in memory, or when Immune isn't running.

### `immune.on_verdict()`

```python
immune.on_verdict(callback: VerdictCallback) -> Callable[[], None]
```

Call `callback(verdict)` for every verdict, on a background thread (it never delays a request). Exceptions in the
callback are logged and ignored.

**Returns** a function that unsubscribes the callback.

**Raises** `immune.ConfigError` when Immune isn't running.

### `immune.on_alert()`

```python
immune.on_alert(callback: AlertCallback) -> None
```

Call `callback(alert)` when a threat's firing rate at a site spikes (see `telemetry.alerts`). An `Alert` has `site`,
`threat`, `fired`, `screened`, `rate`, `baseline_rate`, `at` and a readable `message`.

**Raises** `immune.ConfigError` when Immune isn't running.

## Introspection

### `immune.status()`

```python
immune.status() -> list[SiteStatus]
```

**Returns** a list of `SiteStatus`, one per site Immune has seen, with its archetype, organs, call counts, and enforced
and observed threats. An empty list when Immune isn't running.

### `immune.coverage()`

```python
immune.coverage() -> Coverage
```

**Returns** a `Coverage` with `points` (the patched transports), `adapters` (extra integrations installed) and
`warnings` (anything Immune could see it can't cover).

### `immune.runtime()`

```python
immune.runtime() -> Runtime | None
```

**Returns** the running `Runtime`, or `None`. Advanced use: `runtime.settings` holds the resolved `Settings`.

## Types

### `immune.Verdict`

A frozen dataclass.

| Field | Type | Description |
| --- | --- | --- |
| `trace_id` | `str` | Unique id of the call |
| `site` | `str` | Site name |
| `session_id` | `str \| None` | Your session id; `None` for sessions Immune derived |
| `action` | `Action` | What Immune did |
| `would_action` | `Action` | What it would have done with every hit enforced |
| `hits` | `tuple[Hit, ...]` | Every threat found; default `()` |
| `echo` | `Mapping[str, Mapping[str, float]]` | Schema-echo distributions per structured field; default `{}` |
| `taint` | `Taint` | Session taint; default `Taint.CLEAN` |
| `session_risk` | `float` | 0–1; default `0.0` |
| `sensor` | `SensorInfo` | Jev usage; default `SensorInfo(name="tier0_only")` |
| `config_hash` | `str` | Digest of the settings used |
| `spec_version` | `str` | Threat catalog version, such as `2026.10.1` |
| `explanation` | `str` | One readable line |

| Member | Returns | Description |
| --- | --- | --- |
| `blocked` | `bool` | `True` when `action` changed what the app received beyond a note |
| `enforced_hits` | `tuple[Hit, ...]` | Hits Immune acted on |
| `threats()` | `set[str]` | Threat ids of all hits |
| `to_dict()` | `dict` | JSON-ready representation |
| `Verdict.from_dict(data)` | `Verdict` | The inverse of `to_dict()` |

### `immune.Hit`

| Field | Type | Description |
| --- | --- | --- |
| `threat` | `str` | Threat id |
| `invariant` | `str` | `U1`–`U10`, or `U0` for vaccines |
| `stage` | `Stage` | Where it was found |
| `probability` | `float` | Calibrated probability |
| `action` | `Action` | The action for this threat at this sink |
| `enforced` | `bool` | Whether Immune acted on it |
| `evidence` | `tuple[str, ...]` | Reflex findings and Jev answers, masked; default `()` |
| `frameworks` | `tuple[str, ...]` | OWASP ids; default `()` |
| `organ` | `str \| None` | Owning organ; default `None` |
| `floor` | `str \| None` | `F1`–`F10` for floor rules; default `None` |

### Enumerations

All are `StrEnum`s, so they compare equal to their string values (`verdict.action == "allow"`).

| Enum | Values |
| --- | --- |
| `immune.Action` | `allow`, `annotate`, `augment`, `confirm`, `hold`, `deny`, `neutralize`, `refuse`, `redirect`, `rewrite`, `handoff`, `end_session`. `.severity: int`, `.intervenes: bool`, `Action.most_severe(actions)` |
| `immune.Mode` | `auto`, `observe`, `strict`, `off` |
| `immune.Stage` | `operator`, `input`, `data`, `tool`, `output` |
| `immune.Taint` | `clean`, `external`, `suspicious` |

### `immune.SensorInfo`

`name` (`"jev"`, `"mock"`, `"replay"` or `"tier0_only"`), `model`, `latency_ms`, `input_tokens` and `calls` (Jev
requests made for this call; `0` when answers came from cache).

### `immune.SiteStatus`

`site`, `archetype`, `organs`, `calls`, `enforced`, `observed`, `posture` (risk notes) and `anonymous_calls`.

## Exceptions

| Exception | Raised when |
| --- | --- |
| `immune.ImmuneError` | Base class of all Immune exceptions |
| `immune.ConfigError` | Invalid settings or vaccines, API misuse (such as `configure` before `init`), unknown trace ids in `feedback` |
| `immune.Blocked` | A call was blocked and `on_block: raise` is set. `error.verdict` holds the verdict. For OpenAI clients it also subclasses `openai.OpenAIError`; for Anthropic, `anthropic.AnthropicError` |

## Testing: `immune.testing`

### `immune.testing.ImmuneHarness()`

```python
immune.testing.ImmuneHarness(
    state_dir: Path,
    sensor: Sensor | None = None,
    script: Script | list[FakeReply] | FakeReply | None = None,
    mode: Mode | str | None = None,
    config: dict[str, Any] | str | Path | None = None,
    status: int = 200,
    backend: StateBackend | None = None
)
```

A complete Immune runtime with a scripted model behind it. Doesn't patch the process and doesn't touch the network.

| Parameter | Default | Description |
| --- | --- | --- |
| `state_dir` | required | A folder for this harness's state |
| `sensor` | `None` | A `Sensor`; `None` means `MockSensor()` |
| `script` | `None` | A `FakeReply`, a list played in order, or a function `conversation -> FakeReply` |
| `mode`, `config` | `None` | As for `immune.init()`; `config` may be a path to your `immune.yaml` |
| `status` | `200` | HTTP status the fake provider returns |

| Member | Description |
| --- | --- |
| `openai(asynchronous=False, base_url=...)` | An `OpenAI` (or `AsyncOpenAI`) client through the harness |
| `anthropic(asynchronous=False)`, `gemini()` | Anthropic and Google GenAI clients |
| `http_client()`, `async_http_client()` | `httpx2` clients, for `ChatOpenAI(http_client=...)` and other frameworks |
| `verdict(subject=None)` | Like `immune.verdict`, for this harness |
| `provider` | The `FakeProvider`: `.requests`, `.last_request` show what the model received |
| `runtime`, `settings`, `sensor` | The runtime, its settings and the sensor |
| `close()` | Release state; call it when done |

### `immune.testing.FakeReply()`

```python
immune.testing.FakeReply(text: str = "", tool_calls: Sequence[FakeToolCall] = ())
```

What the scripted model answers: text, tool calls, or both.

### `immune.testing.FakeToolCall()`

```python
immune.testing.FakeToolCall(
    name: str,
    arguments: Mapping[str, Any] = {},
    call_id: str = <generated>
)
```

One tool call in a `FakeReply`. `call_id` is generated when omitted.

### `immune.testing.FakeProvider()`

```python
immune.testing.FakeProvider(
    script: Script | Sequence[FakeReply] | FakeReply | None = None,
    status: int = 200,
    delta_chars: int = 8
)
```

A fake OpenAI, Anthropic and Gemini server behind an `httpx` transport: `provider.transport(httpx2)`. Streams are
split into chunks of `delta_chars` characters.

### `immune.testing.MockSensor()`

```python
immune.testing.MockSensor(
    signals: Mapping[str, ScriptedValue] | None = None,
    rules: Sequence[Rule] = (),
    default_noul: float = 0.02,
    fail: bool = False
)
```

A scripted Jev. Answers by question key; `immune explain <threat>` lists the keys.

| Parameter | Default | Description |
| --- | --- | --- |
| `signals` | `None` | `{key: value}`: a probability for yes/no questions, an option name or `{option: p}` for choices |
| `rules` | `()` | Functions `(state, question) -> value or None`, consulted first |
| `default_noul` | `0.02` | The answer for unscripted yes/no questions; candidate questions default to `1 - default_noul` (Jev agrees with the reflex) |
| `fail` | `False` | Raise `SensorUnavailable` on every request, to simulate an outage |

`sensor.script(**signals)` changes answers later; `sensor.calls` records every request as `(state, question_keys)`.

### `immune.testing.RecordingSensor()`

```python
immune.testing.RecordingSensor(inner: Sensor, path: Path)
```

Wraps a real sensor and appends each answer to a JSON Lines cassette at `path`.

### `immune.testing.ReplaySensor()`

```python
immune.testing.ReplaySensor(path: Path)
```

Serves answers from a cassette. A request that wasn't recorded raises `SensorUnavailable`, which Immune treats as an
outage.

### `immune.testing.ScenarioLibrary()`

```python
immune.testing.ScenarioLibrary(roots: list[Path] | None = None)
```

Loads scenario files. Default roots: `./scenarios` and Immune's own incident library. `.all()` lists scenarios,
`.find(id)` returns one (raises `ConfigError` if unknown), and `ScenarioLibrary.load(path)` reads a file.

### `immune.testing.ScenarioRunner()`

```python
immune.testing.ScenarioRunner(
    state_dir: Path | None = None,
    sensor_factory: SensorFactory | None = None
)
```

Runs a scenario through a fresh harness per scenario. `sensor_factory(scenario)` defaults to
`MockSensor(scenario.signals)`. `.run(scenario)` returns a `ScenarioResult` with `passed`, `failures` and `turns`
(each with `reply`, `verdict` and `failures`).

### `immune.testing.LangSmithRecorder()`

```python
immune.testing.LangSmithRecorder()
```

A context manager standing in for the LangSmith API. Inside it, Immune sends runs and feedback to the recorder:
`recorder.runs`, `recorder.immune_runs()`, `recorder.children(run)`, `recorder.feedback` and
`recorder.feedback_on(run_id, key)`.

## Command line

The `immune` command is installed with the package. (In a terminal, drop the `!`.)

In [2]:
!immune --help

usage: immune [-h] [-v]
              {test,replay,threats,explain,status,posture,promote,label,calibrate,evidence,vaccines,vaccinate,export,init,config,doctor,version} ...

A general immune system for LLM applications.

positional arguments:
  {test,replay,threats,explain,status,posture,promote,label,calibrate,evidence,vaccines,vaccinate,export,init,config,doctor,version}
    test                screen one message (and optionally a draft reply) and
                        print the verdict
    replay              replay incident, chain and benign scenarios offline
    threats             list the threat catalog
    explain             show how a threat is detected and handled
    status              show call sites, profiles and promotions recorded in
                        the state directory
    posture             list configuration risks for every observed call site
    promote             show which observed defenses have earned automatic
                        enforcement
    

Every command has `--help`. The ones you'll use most: `immune doctor --live`, `immune config validate`,
`immune explain <threat>`, `immune test "<message>" --reply "<draft>"`, `immune vaccines test|trial|list`,
`immune vaccinate`, `immune replay`, `immune status`.